In [12]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
from torch.utils.data import Dataset, DataLoader

import config
from data.london_generator import generate_london_graph
from data.dataset import LondonTSPDataset
from solvers.nearest_neighbor import solve_nn
from solvers.two_opt import solve_two_opt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [13]:
import random

class LondonCostDataset(Dataset):
    def __init__(self, size, n_nodes):
        self.size = size
        self.n_nodes = n_nodes

    def __len__(self):
        return self.size

    def __getitem__(self, idx):

        n_bridges = random.randint(*config.N_BRIDGES_TRAIN)

        nodes, bridges, C = generate_london_graph(
            n_nodes=self.n_nodes,
            n_bridges=n_bridges,
            congestion_strength=config.CONGESTION_STRENGTH,
            bridge_penalty=config.BRIDGE_PENALTY,
            asymmetry_strength=config.ASYMMETRY_STRENGTH
        )

        return torch.tensor(C, dtype=torch.float32)

In [14]:
class CostTSPModel(nn.Module):
    def __init__(self, n_nodes, embed_dim=128, n_heads=4, n_layers=2):
        super().__init__()

        self.input_proj = nn.Linear(n_nodes, embed_dim)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=n_heads,
            batch_first=True
        )

        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=n_layers
        )

        self.value_head = nn.Sequential(
            nn.Linear(embed_dim, embed_dim),
            nn.ReLU(),
            nn.Linear(embed_dim, 1)
        )

    def forward(self, C):
        # Normalize per graph
        C_norm = C / (C.max(dim=-1, keepdim=True)[0] + 1e-8)
        x = self.input_proj(C_norm)
        encoded = self.encoder(x)
        return encoded

    def decode(self, encoded, current, visited):
        query = encoded[torch.arange(encoded.size(0)), current]
        scores = torch.matmul(query.unsqueeze(1), encoded.transpose(1,2)).squeeze(1)
        scores = scores.masked_fill(visited, -1e9)
        return torch.softmax(scores, dim=-1)

    def value(self, encoded):
        graph_embed = encoded.mean(dim=1)
        return self.value_head(graph_embed).squeeze(-1)

In [15]:
def sample_tour(model, C):

    encoded = model(C)
    values = model.value(encoded)

    batch, n, _ = C.size()
    visited = torch.zeros(batch, n, dtype=torch.bool, device=C.device)
    current = torch.zeros(batch, dtype=torch.long, device=C.device)

    log_probs = []
    total_rewards = torch.zeros(batch, device=C.device)

    for _ in range(n - 1):

        probs = model.decode(encoded, current, visited)
        dist = torch.distributions.Categorical(probs)
        nxt = dist.sample()

        log_probs.append(dist.log_prob(nxt))

        edge_cost = C[torch.arange(batch), current, nxt]
        total_rewards += -edge_cost

        visited = visited.clone()
        visited[torch.arange(batch), nxt] = True
        current = nxt

    log_probs = torch.stack(log_probs).sum(0)

    return log_probs, total_rewards, values

In [16]:
def train_model(n_nodes=50, epochs=300, batch_size=64):

    model = CostTSPModel(n_nodes).to(device)
    optimizer = optim.Adam(model.parameters(), lr=3e-5)

    dataset = LondonCostDataset(size=1000, n_nodes=n_nodes)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    critic_weight = 0.1

    for epoch in range(epochs):

        epoch_cost = 0

        for C in loader:

            C = C.to(device)

            log_probs, rewards, values = sample_tour(model, C)

            costs = -rewards.detach()

            reward_norm = (rewards - rewards.mean()) / (rewards.std() + 1e-8)
            advantage = reward_norm - values
            advantage = (advantage - advantage.mean()) / (advantage.std() + 1e-8)

            actor_loss = -(advantage.detach() * log_probs).mean()
            critic_loss = F.mse_loss(values, reward_norm.detach())

            loss = actor_loss + critic_weight * critic_loss

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            epoch_cost += costs.mean().item()

        print(f"Epoch {epoch+1}/{epochs} | Avg Cost: {epoch_cost/len(loader):.2f}")

    return model

In [17]:
def greedy_decode(model, C):

    model.eval()

    with torch.no_grad():

        C = torch.as_tensor(C, dtype=torch.float32, device=device).unsqueeze(0)
        encoded = model(C)

        n = C.size(1)
        visited = torch.zeros(1, n, dtype=torch.bool, device=device)
        current = torch.zeros(1, dtype=torch.long, device=device)

        visited[0, 0] = True  # mark start visited

        tour = [0]
        total_cost = 0

        for _ in range(n - 1):

            probs = model.decode(encoded, current, visited)
            nxt = torch.argmax(probs, dim=-1)

            total_cost += C[0, current.item(), nxt.item()].item()

            visited[0, nxt] = True
            current = nxt
            tour.append(nxt.item())

        # 🔥 CLOSE THE TOUR
        total_cost += C[0, current.item(), 0].item()
        tour.append(0)

    return tour, total_cost

In [18]:
def evaluate_model(model, n_nodes=50, n_instances=20):

    results = []

    for _ in range(n_instances):

        nodes, bridges, C = generate_london_graph(
            n_nodes=n_nodes,
            n_bridges=4,
            congestion_strength=config.CONGESTION_STRENGTH,
            bridge_penalty=config.BRIDGE_PENALTY,
            asymmetry_strength=config.ASYMMETRY_STRENGTH
        )

        C_tensor = torch.tensor(C, dtype=torch.float32).unsqueeze(0).to(device)

        # RL
        tour, rl_cost = greedy_decode(model, C)

        # NN
        _, nn_cost = solve_nn(C)

        # 2-opt
        nn_tour, _ = solve_nn(C)
        _, opt_cost = solve_two_opt(nn_tour, C)

        results.append([rl_cost, nn_cost, opt_cost])

    results = np.array(results)

    print("RL Mean:", results[:,0].mean())
    print("NN Mean:", results[:,1].mean())
    print("2-opt Mean:", results[:,2].mean())

In [19]:
model = train_model(n_nodes=50, epochs=300)

Epoch 1/300 | Avg Cost: 1431.34
Epoch 2/300 | Avg Cost: 1333.27
Epoch 3/300 | Avg Cost: 1276.72
Epoch 4/300 | Avg Cost: 1232.25
Epoch 5/300 | Avg Cost: 1207.51
Epoch 6/300 | Avg Cost: 1187.19
Epoch 7/300 | Avg Cost: 1177.05
Epoch 8/300 | Avg Cost: 1169.71
Epoch 9/300 | Avg Cost: 1160.70
Epoch 10/300 | Avg Cost: 1151.63
Epoch 11/300 | Avg Cost: 1141.61
Epoch 12/300 | Avg Cost: 1137.66
Epoch 13/300 | Avg Cost: 1137.23
Epoch 14/300 | Avg Cost: 1125.37
Epoch 15/300 | Avg Cost: 1123.22
Epoch 16/300 | Avg Cost: 1122.42
Epoch 17/300 | Avg Cost: 1118.91
Epoch 18/300 | Avg Cost: 1109.25
Epoch 19/300 | Avg Cost: 1111.27
Epoch 20/300 | Avg Cost: 1109.83
Epoch 21/300 | Avg Cost: 1111.43
Epoch 22/300 | Avg Cost: 1107.86
Epoch 23/300 | Avg Cost: 1106.67
Epoch 24/300 | Avg Cost: 1103.80
Epoch 25/300 | Avg Cost: 1098.31
Epoch 26/300 | Avg Cost: 1099.39
Epoch 27/300 | Avg Cost: 1100.84
Epoch 28/300 | Avg Cost: 1097.44
Epoch 29/300 | Avg Cost: 1092.39
Epoch 30/300 | Avg Cost: 1091.76
Epoch 31/300 | Avg 

In [22]:
evaluate_model(model)

RL Mean: 1080.8336631260813
NN Mean: 1100.3866248992758
2-opt Mean: 1013.7648690986334


In [23]:
torch.save(model.state_dict(), "cost_model_final.pt")